<a href="https://colab.research.google.com/github/TienPhap0102/06_SmartToys-Ecommerce-Profitability-Customer-Journey-Analytics/blob/main/01_run_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **1. Đẩy dữ liệu raw lên Big Query**

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import pandas as pd
from pathlib import Path
from google.cloud import bigquery
from google.oauth2 import service_account
from google.api_core.exceptions import NotFound

# Bước 1: Khai báo sáu file dữ liệu raw
raw_dir = Path("/content/drive/MyDrive/1. Data anh Huy/Big Project Ecommerce/Dataset/bronze")
files = {
    "products": "products.csv",
    "website_sessions": "website_sessions.csv",
    "website_pageviews": "website_pageviews.csv",
    "orders": "orders.csv",
    "order_items": "order_items.csv",
    "order_item_refunds": "order_item_refunds.csv"
}

# Bước 2: Thông tin kết nối BigQuery
project_id = "jda-k1"
raw_dataset_id = "bigproject_phap_raw"
key_path = Path("/content/drive/MyDrive/1. Data anh Huy/Big Project Ecommerce/Dataset/jda-k1-2afad6d8f47e.json")


# Bước 3: Tạo client BigQuery bằng service account
credentials = service_account.Credentials.from_service_account_file(str(key_path))
client = bigquery.Client(credentials=credentials, project=project_id)

# Chuẩn bị ba dataset
dataset_ids = [
    "bigproject_phap_raw",
    "bigproject_phap_staging",
    "bigproject_phap_mart"
]

dataset_locations = {}

for dataset_id in dataset_ids:
    dataset_ref = f"{project_id}.{dataset_id}"

    dataset = bigquery.Dataset(dataset_ref)
    dataset.location = "US"

    dataset = client.create_dataset(dataset, exists_ok=True)
    dataset_locations[dataset_id] = dataset.location

    print(f"Dataset sẵn sàng: {dataset_ref} | {dataset.location}")

# Các tầng cần cùng location để truy vấn kết hợp
if len(set(dataset_locations.values())) != 1:
    raise ValueError("Ba dataset đang khác location; cần kiểm tra trước.")

raw_location = dataset_locations[raw_dataset_id]

Dataset sẵn sàng: jda-k1.bigproject_phap_raw | US
Dataset sẵn sàng: jda-k1.bigproject_phap_staging | US
Dataset sẵn sàng: jda-k1.bigproject_phap_mart | US


In [ ]:
upload_log = []

for table_id, filename in files.items():
    table_ref = f"{project_id}.{raw_dataset_id}.{table_id}"

    # Đọc CSV: tự nhận kiểu dữ liệu và giá trị thiếu
    df = pd.read_csv(
        raw_dir / filename,
        encoding="utf-8-sig",
        low_memory=False
    )

    # Thay bảng raw cũ, suy ra schema từ DataFrame
    job_config = bigquery.LoadJobConfig(
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
    )

    # Nạp dữ liệu lên BigQuery
    job = client.load_table_from_dataframe(
        df,
        table_ref,
        job_config=job_config,
        location=raw_location
    )
    job.result()

    # Kiểm tra số dòng sau khi nạp
    loaded_table = client.get_table(table_ref)

    if loaded_table.num_rows != len(df):
        raise ValueError(
            f"Số dòng không khớp ở {table_id}: "
            f"CSV={len(df):,}, BigQuery={loaded_table.num_rows:,}"
        )

    upload_log.append({
        "table": table_id,
        "status": "OK",
        "source_rows": len(df),
        "bigquery_rows": loaded_table.num_rows
    })

    print(f"OK — {table_id}: {loaded_table.num_rows:,} dòng")
    print(
        "Schema:",
        ", ".join(
            f"{field.name}: {field.field_type}"
            for field in loaded_table.schema
        )
    )

display(pd.DataFrame(upload_log))

OK — products: 4 dòng
Schema: product_id: INTEGER, created_at: STRING, product_name: STRING
OK — website_sessions: 472,871 dòng
Schema: website_session_id: INTEGER, created_at: STRING, user_id: INTEGER, is_repeat_session: INTEGER, utm_source: STRING, utm_campaign: STRING, utm_content: STRING, device_type: STRING, http_referer: STRING
OK — website_pageviews: 1,188,124 dòng
Schema: website_pageview_id: INTEGER, created_at: STRING, website_session_id: INTEGER, pageview_url: STRING
OK — orders: 32,313 dòng
Schema: order_id: INTEGER, created_at: STRING, website_session_id: INTEGER, user_id: INTEGER, primary_product_id: INTEGER, items_purchased: INTEGER, price_usd: FLOAT, cogs_usd: FLOAT
OK — order_items: 40,025 dòng
Schema: order_item_id: INTEGER, created_at: STRING, order_id: INTEGER, product_id: INTEGER, is_primary_item: INTEGER, price_usd: FLOAT, cogs_usd: FLOAT
OK — order_item_refunds: 1,731 dòng
Schema: order_item_refund_id: INTEGER, created_at: STRING, order_item_id: INTEGER, order_id

,table,status,source_rows,bigquery_rows
0,products,OK,4,4
1,website_sessions,OK,472871,472871
2,website_pageviews,OK,1188124,1188124
3,orders,OK,32313,32313
4,order_items,OK,40025,40025
5,order_item_refunds,OK,1731,1731


# **2. Lấy dữ liệu raw từ Big Query**

In [ ]:
%pip install -q google-cloud-bigquery pandas pyarrow db-dtypes

In [ ]:
import pandas as pd
from google.cloud import bigquery
from google.oauth2 import service_account

project_id = "jda-k1"
dataset_id = "bigproject_phap_raw"
key_path = "/content/drive/MyDrive/1. Data anh Huy/Big Project Ecommerce/Dataset/jda-k1-2afad6d8f47e.json"

credentials = service_account.Credentials.from_service_account_file(
    key_path
)

client = bigquery.Client(
    credentials=credentials,
    project=project_id
)

In [ ]:
%pip install -q db-dtypes

table_names = [
    "products",
    "website_sessions",
    "website_pageviews",
    "orders",
    "order_items",
    "order_item_refunds"
]

raw_tables = {}

for table_name in table_names:
    query = f"""
    SELECT *
    FROM `jda-k1.bigproject_phap_raw.{table_name}`
    """

    raw_tables[table_name] = (
        client.query(query)
        .to_dataframe(create_bqstorage_client=False)
    )

    print(f"{table_name}: {len(raw_tables[table_name]):,} dòng")

products: 4 dòng
website_sessions: 472,871 dòng
website_pageviews: 1,188,124 dòng
orders: 32,313 dòng
order_items: 40,025 dòng
order_item_refunds: 1,731 dòng


In [ ]:
raw_tables['products'].head()

,product_id,created_at,product_name
0,1,2012-03-19 08:00:00,The Original Mr. Fuzzy
1,2,2013-01-06 13:00:00,The Forever Love Bear
2,3,2013-12-12 09:00:00,The Birthday Sugar Panda
3,4,2014-02-05 10:00:00,The Hudson River Mini bear


# **3. Data Processing and Cleaning**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

### **3.1. Products table**

In [ ]:
# Data overview
raw_tables['products'].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   product_id    4 non-null      Int64 
 1   created_at    4 non-null      object
 2   product_name  4 non-null      object
dtypes: Int64(1), object(2)
memory usage: 232.0+ bytes


In [ ]:
# Check null
raw_tables['products'].isnull().sum()

,0
product_id,0
created_at,0
product_name,0


In [ ]:
# Check duplicate
raw_tables['products'].duplicated().sum()

np.int64(0)

**Observation for Products Table**

*   **Data Overview**: The table contains 4 entries and 3 columns (`product_id`, `created_at`, `product_name`).
*   **Data Types**: `product_id` is an integer (`int64`), while `created_at` and `product_name` are objects, indicating they are likely strings or datetime objects that could be converted for further analysis.
*   **Missing Values**: There are no null values in any of the columns, which is excellent.
*   **Duplicate Rows**: There are no duplicate rows in the `products` table, ensuring data uniqueness and integrity.

**Data cleaning**

In [ ]:
# Transform data format of date column
raw_tables['products']['created_at'] = raw_tables['products']['created_at'].astype('datetime64[ns]')

In [ ]:
raw_tables['products'].info()
print('Products clean!!!')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   product_id    4 non-null      Int64         
 1   created_at    4 non-null      datetime64[ns]
 2   product_name  4 non-null      object        
dtypes: Int64(1), datetime64[ns](1), object(1)
memory usage: 232.0+ bytes
Products clean!!!


In [ ]:
products_clean = raw_tables['products'].copy()
products_clean.head()

,product_id,created_at,product_name
0,1,2012-03-19 08:00:00,The Original Mr. Fuzzy
1,2,2013-01-06 13:00:00,The Forever Love Bear
2,3,2013-12-12 09:00:00,The Birthday Sugar Panda
3,4,2014-02-05 10:00:00,The Hudson River Mini bear


In [ ]:
products_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   product_id    4 non-null      Int64         
 1   created_at    4 non-null      datetime64[ns]
 2   product_name  4 non-null      object        
dtypes: Int64(1), datetime64[ns](1), object(1)
memory usage: 232.0+ bytes


### **3.2. Order items table**

In [ ]:
raw_tables['order_items'].head()

,order_item_id,created_at,order_id,product_id,is_primary_item,price_usd,cogs_usd
0,11466,2014-02-05 11:02:39,11148,4,0,29.99,9.49
1,11468,2014-02-05 12:15:56,11149,4,0,29.99,9.49
2,11470,2014-02-05 13:07:28,11150,4,0,29.99,9.49
3,11473,2014-02-05 13:32:46,11152,4,0,29.99,9.49
4,11475,2014-02-05 13:45:17,11153,4,0,29.99,9.49


In [ ]:
# Overview
raw_tables['order_items'].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40025 entries, 0 to 40024
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   order_item_id    40025 non-null  Int64  
 1   created_at       40025 non-null  object 
 2   order_id         40025 non-null  Int64  
 3   product_id       40025 non-null  Int64  
 4   is_primary_item  40025 non-null  Int64  
 5   price_usd        40025 non-null  float64
 6   cogs_usd         40025 non-null  float64
dtypes: Int64(4), float64(2), object(1)
memory usage: 2.3+ MB


In [ ]:
# Check null
raw_tables['order_items'].isnull().sum()

,0
order_item_id,0
created_at,0
order_id,0
product_id,0
is_primary_item,0
price_usd,0
cogs_usd,0


In [ ]:
# Check duplicate
raw_tables['order_items'].duplicated().sum()

np.int64(0)

In [ ]:
# Describe
raw_tables['order_items'].describe()

,order_item_id,order_id,product_id,is_primary_item,price_usd,cogs_usd
count,40025.0,40025.0,40025.0,40025.0,40025.000000,40025.000000
mean,20013.0,17121.957502,1.770019,0.80732,48.432473,18.047976
std,11554.366599,9053.765867,1.085561,0.394408,8.012370,3.856821
min,1.0,1.0,1.0,0.0,29.990000,9.490000
25%,10007.0,9871.0,1.0,1.0,49.990000,19.490000
50%,20013.0,17490.0,1.0,1.0,49.990000,19.490000
75%,30019.0,24818.0,2.0,1.0,49.990000,19.490000
max,40025.0,32313.0,4.0,1.0,59.990000,22.490000


**Observation for Products Table**

*   **Data Overview**: The table contains 4 entries and 3 columns (`product_id`, `created_at`, `product_name`).
*   **Data Types**: `product_id` is an integer (`int64`), while `created_at` and `product_name` are objects, indicating they are likely strings or datetime objects that could be converted for further analysis.
*   **Missing Values**: There are no null values in any of the columns, which is excellent.
*   **Duplicate Rows**: There are no duplicate rows in the `products` table, ensuring data uniqueness and integrity.

**Data cleaning**

In [ ]:
# Convert data format created_at
raw_tables['order_items']['created_at'] = raw_tables['order_items']['created_at'].astype('datetime64[ns]')

In [ ]:
raw_tables['order_items'].info()
print('Order items clean!!!')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40025 entries, 0 to 40024
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   order_item_id    40025 non-null  Int64         
 1   created_at       40025 non-null  datetime64[ns]
 2   order_id         40025 non-null  Int64         
 3   product_id       40025 non-null  Int64         
 4   is_primary_item  40025 non-null  Int64         
 5   price_usd        40025 non-null  float64       
 6   cogs_usd         40025 non-null  float64       
dtypes: Int64(4), datetime64[ns](1), float64(2)
memory usage: 2.3 MB
Order items clean!!!


In [ ]:
order_items_clean = raw_tables['order_items'].copy()
order_items_clean.head()

,order_item_id,created_at,order_id,product_id,is_primary_item,price_usd,cogs_usd
0,11466,2014-02-05 11:02:39,11148,4,0,29.99,9.49
1,11468,2014-02-05 12:15:56,11149,4,0,29.99,9.49
2,11470,2014-02-05 13:07:28,11150,4,0,29.99,9.49
3,11473,2014-02-05 13:32:46,11152,4,0,29.99,9.49
4,11475,2014-02-05 13:45:17,11153,4,0,29.99,9.49


### **3.3. Orders table**

In [ ]:
raw_tables['orders'].head()

,order_id,created_at,website_session_id,user_id,primary_product_id,items_purchased,price_usd,cogs_usd
0,1,2012-03-19 10:42:46,20,20,1,1,49.99,19.49
1,2,2012-03-19 19:27:37,104,104,1,1,49.99,19.49
2,3,2012-03-20 06:44:45,147,147,1,1,49.99,19.49
3,4,2012-03-20 09:41:45,160,160,1,1,49.99,19.49
4,5,2012-03-20 11:28:15,177,177,1,1,49.99,19.49


In [ ]:
# Overview
raw_tables['orders'].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32313 entries, 0 to 32312
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   order_id            32313 non-null  Int64  
 1   created_at          32313 non-null  object 
 2   website_session_id  32313 non-null  Int64  
 3   user_id             32313 non-null  Int64  
 4   primary_product_id  32313 non-null  Int64  
 5   items_purchased     32313 non-null  Int64  
 6   price_usd           32313 non-null  float64
 7   cogs_usd            32313 non-null  float64
dtypes: Int64(5), float64(2), object(1)
memory usage: 2.1+ MB


In [ ]:
# check null
raw_tables['orders'].isnull().sum()

,0
order_id,0
created_at,0
website_session_id,0
user_id,0
primary_product_id,0
items_purchased,0
price_usd,0
cogs_usd,0


In [ ]:
# check dupicate
raw_tables['orders'].duplicated().sum()

np.int64(0)

In [ ]:
# describe
raw_tables['orders'].describe()

,order_id,website_session_id,user_id,primary_product_id,items_purchased,price_usd,cogs_usd
count,32313.0,32313.0,32313.0,32313.0,32313.0,32313.000000,32313.000000
mean,16157.0,258292.288707,215691.622629,1.392474,1.238666,59.991636,22.355406
std,9328.103961,132427.649842,108402.203189,0.732277,0.426274,17.808771,6.238621
min,1.0,20.0,13.0,1.0,1.0,29.990000,9.490000
25%,8079.0,144828.0,124135.0,1.0,1.0,49.990000,19.490000
50%,16157.0,263554.0,221461.0,1.0,1.0,49.990000,19.490000
75%,24235.0,374799.0,310542.0,2.0,1.0,59.990000,22.490000
max,32313.0,472818.0,394273.0,4.0,2.0,109.980000,41.980000


**Observation for Orders Table**

Based on the analysis of the `orders` table, we can make the following observations:

*   **Data Overview**: The table contains 32,313 entries and 8 columns, including `order_id`, `created_at`, `website_session_id`, `user_id`, `primary_product_id`, `items_purchased`, `price_usd`, and `cogs_usd`.
*   **Data Types**: `order_id`, `website_session_id`, `user_id`, `primary_product_id`, and `items_purchased` are integers (`int64`). `price_usd` and `cogs_usd` are floating-point numbers (`float64`). The `created_at` column is an object type, which should be converted to a datetime object for proper time-series analysis.
*   **Missing Values**: There are no null values across all columns, indicating a complete dataset for `orders`.
*   **Duplicate Rows**: The `orders` table contains no duplicate rows, ensuring data integrity and uniqueness of each order record.
*   **Descriptive Statistics**:
    *   `items_purchased` mostly shows 1 or 2 items per order (mean ~1.24), indicating customers generally purchase a small number of items per transaction.
    *   `price_usd` ranges from 29.99 to 109.98, with an average of approximately 59.99 USD.
    *   `cogs_usd` (Cost of Goods Sold) ranges from 9.49 to 41.98, with an average of about 22.35 USD.

In summary, the `orders` table is clean regarding missing values and duplicates. The `created_at` column type conversion is a necessary step for further temporal analysis.

In [ ]:
# Conver created_at to datetime
raw_tables['orders']['created_at'] = raw_tables['orders']['created_at'].astype('datetime64[ns]')

In [ ]:
raw_tables['orders'].info()
print('Orders clean!!!')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32313 entries, 0 to 32312
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   order_id            32313 non-null  Int64         
 1   created_at          32313 non-null  datetime64[ns]
 2   website_session_id  32313 non-null  Int64         
 3   user_id             32313 non-null  Int64         
 4   primary_product_id  32313 non-null  Int64         
 5   items_purchased     32313 non-null  Int64         
 6   price_usd           32313 non-null  float64       
 7   cogs_usd            32313 non-null  float64       
dtypes: Int64(5), datetime64[ns](1), float64(2)
memory usage: 2.1 MB
Orders clean!!!


In [ ]:
orders_clean = raw_tables['orders'].copy()
orders_clean.head()

,order_id,created_at,website_session_id,user_id,primary_product_id,items_purchased,price_usd,cogs_usd
0,1,2012-03-19 10:42:46,20,20,1,1,49.99,19.49
1,2,2012-03-19 19:27:37,104,104,1,1,49.99,19.49
2,3,2012-03-20 06:44:45,147,147,1,1,49.99,19.49
3,4,2012-03-20 09:41:45,160,160,1,1,49.99,19.49
4,5,2012-03-20 11:28:15,177,177,1,1,49.99,19.49


### **3.4. Order item refunds table**

In [ ]:
raw_tables['order_item_refunds'].head()

,order_item_refund_id,created_at,order_item_id,order_id,refund_amount_usd
0,576,2014-02-17 22:42:45,11493,11165,29.99
1,596,2014-03-01 20:14:03,12430,11866,29.99
2,629,2014-03-25 18:40:12,13230,12474,29.99
3,699,2014-05-04 07:32:54,15518,14172,29.99
4,702,2014-05-05 13:42:18,15266,13994,29.99


In [ ]:
# Overview
raw_tables['order_item_refunds'].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1731 entries, 0 to 1730
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   order_item_refund_id  1731 non-null   Int64  
 1   created_at            1731 non-null   object 
 2   order_item_id         1731 non-null   Int64  
 3   order_id              1731 non-null   Int64  
 4   refund_amount_usd     1731 non-null   float64
dtypes: Int64(3), float64(1), object(1)
memory usage: 72.8+ KB


In [ ]:
# Check null
raw_tables['order_item_refunds'].isnull().sum()

,0
order_item_refund_id,0
created_at,0
order_item_id,0
order_id,0
refund_amount_usd,0


In [ ]:
# Check duplicate
raw_tables['order_item_refunds'].duplicated().sum()

np.int64(0)

**Observation for Order Item Refunds Table**

Based on the analysis of the `order_item_refunds` table, we can make the following observations:

*   **Data Overview**: The table contains 1,731 entries and 5 columns, including `order_item_refund_id`, `created_at`, `order_item_id`, `order_id`, and `refund_amount_usd`.
*   **Data Types**: `order_item_refund_id`, `order_item_id`, and `order_id` are integers (`int64`). `refund_amount_usd` is a floating-point number (`float64`). The `created_at` column is an object type, which should be converted to a datetime object for proper time-series analysis.
*   **Missing Values**: There are no null values in any of the columns, indicating a complete and clean dataset for refunds.
*   **Duplicate Rows**: The `order_item_refunds` table contains no duplicate rows, ensuring the uniqueness of each refund record.
*   **Descriptive Statistics**:
    *   `refund_amount_usd` ranges from 29.99 to 59.99, with an average of approximately 49.30 USD.

**Data cleaning**

In [ ]:
# Convert data format created_at
raw_tables['order_item_refunds']['created_at'] = raw_tables['order_item_refunds']['created_at'].astype('datetime64[ns]')

In [ ]:
# Check overview
raw_tables['order_item_refunds'].info()
print('Order item refunds clean!!!')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1731 entries, 0 to 1730
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   order_item_refund_id  1731 non-null   Int64         
 1   created_at            1731 non-null   datetime64[ns]
 2   order_item_id         1731 non-null   Int64         
 3   order_id              1731 non-null   Int64         
 4   refund_amount_usd     1731 non-null   float64       
dtypes: Int64(3), datetime64[ns](1), float64(1)
memory usage: 72.8 KB
Order item refunds clean!!!


In [ ]:
order_items_refunds_clean = raw_tables['order_item_refunds'].copy()
order_items_refunds_clean.head()

,order_item_refund_id,created_at,order_item_id,order_id,refund_amount_usd
0,576,2014-02-17 22:42:45,11493,11165,29.99
1,596,2014-03-01 20:14:03,12430,11866,29.99
2,629,2014-03-25 18:40:12,13230,12474,29.99
3,699,2014-05-04 07:32:54,15518,14172,29.99
4,702,2014-05-05 13:42:18,15266,13994,29.99


### **3.5. Website_pageviews table**

In [ ]:
raw_tables['website_pageviews'].head()

,website_pageview_id,created_at,website_session_id,pageview_url
0,13,2012-03-19 09:19:52,6,/billing
1,36,2012-03-19 10:36:31,20,/billing
2,94,2012-03-19 13:43:13,49,/billing
3,159,2012-03-19 18:11:37,90,/billing
4,176,2012-03-19 19:05:03,100,/billing


In [ ]:
# Overview
raw_tables['website_pageviews'].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1188124 entries, 0 to 1188123
Data columns (total 4 columns):
 #   Column               Non-Null Count    Dtype 
---  ------               --------------    ----- 
 0   website_pageview_id  1188124 non-null  Int64 
 1   created_at           1188124 non-null  object
 2   website_session_id   1188124 non-null  Int64 
 3   pageview_url         1188124 non-null  object
dtypes: Int64(2), object(2)
memory usage: 38.5+ MB


In [ ]:
# Check null
raw_tables['website_pageviews'].isnull().sum()

,0
website_pageview_id,0
created_at,0
website_session_id,0
pageview_url,0


In [ ]:
# Check duplicate
raw_tables['website_pageviews'].duplicated().sum()

np.int64(0)

In [ ]:
# check pageview_url
raw_tables['website_pageviews']['pageview_url'].value_counts()

,count
pageview_url,
/products,261231
/the-original-mr-fuzzy,162525
/home,137576
/lander-2,131170
/cart,94953
/lander-3,79000
/lander-5,68166
/shipping,64484
/billing-2,48441


**Observation for Website Pageviews Table**

Based on the analysis of the `website_pageviews` table, we can make the following observations:

*   **Data Overview**: The table contains 1,188,124 entries and 4 columns: `website_pageview_id`, `created_at`, `website_session_id`, and `pageview_url`.
*   **Data Types**: `website_pageview_id` and `website_session_id` are integers (`int64`). `created_at` and `pageview_url` are object types. The `created_at` column should be converted to a datetime object for time-series analysis.
*   **Missing Values**: There are no null values in any of the columns, indicating a complete and clean dataset.
*   **Duplicate Rows**: The `website_pageviews` table contains no duplicate rows, ensuring the uniqueness of each pageview record.

**Data cleaning**

In [ ]:
# Convert data format created_at
raw_tables['website_pageviews']['created_at'] = raw_tables['website_pageviews']['created_at'].astype('datetime64[ns]')

In [ ]:
raw_tables['website_pageviews'].info()
print('Website pageviews clean!!!')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1188124 entries, 0 to 1188123
Data columns (total 4 columns):
 #   Column               Non-Null Count    Dtype         
---  ------               --------------    -----         
 0   website_pageview_id  1188124 non-null  Int64         
 1   created_at           1188124 non-null  datetime64[ns]
 2   website_session_id   1188124 non-null  Int64         
 3   pageview_url         1188124 non-null  object        
dtypes: Int64(2), datetime64[ns](1), object(1)
memory usage: 38.5+ MB
Website pageviews clean!!!


In [ ]:
website_pageviews_clean = raw_tables['website_pageviews'].copy()
website_pageviews_clean.head()

,website_pageview_id,created_at,website_session_id,pageview_url
0,13,2012-03-19 09:19:52,6,/billing
1,36,2012-03-19 10:36:31,20,/billing
2,94,2012-03-19 13:43:13,49,/billing
3,159,2012-03-19 18:11:37,90,/billing
4,176,2012-03-19 19:05:03,100,/billing


### **3.6. Website_sessions table**

In [ ]:
raw_tables['website_sessions'].head()

,website_session_id,created_at,user_id,is_repeat_session,utm_source,utm_campaign,utm_content,device_type,http_referer
0,934,2012-03-25 14:23:48,934,0,None,None,None,desktop,None
1,1045,2012-03-26 11:54:49,846,1,None,None,None,desktop,None
2,1051,2012-03-26 12:48:41,152,1,None,None,None,desktop,None
3,1147,2012-03-27 00:44:11,1143,0,None,None,None,desktop,None
4,1248,2012-03-27 15:32:30,1243,0,None,None,None,desktop,https://www.gsearch.com


In [ ]:
# Overivew
raw_tables['website_sessions'].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 472871 entries, 0 to 472870
Data columns (total 9 columns):
 #   Column              Non-Null Count   Dtype 
---  ------              --------------   ----- 
 0   website_session_id  472871 non-null  Int64 
 1   created_at          472871 non-null  object
 2   user_id             472871 non-null  Int64 
 3   is_repeat_session   472871 non-null  Int64 
 4   utm_source          389543 non-null  object
 5   utm_campaign        389543 non-null  object
 6   utm_content         389543 non-null  object
 7   device_type         472871 non-null  object
 8   http_referer        432954 non-null  object
dtypes: Int64(3), object(6)
memory usage: 33.8+ MB


In [ ]:
# Check null
raw_tables['website_sessions'].isnull().sum()

,0
website_session_id,0
created_at,0
user_id,0
is_repeat_session,0
utm_source,83328
utm_campaign,83328
utm_content,83328
device_type,0
http_referer,39917


In [ ]:
# Check duplicate
raw_tables['website_sessions'].duplicated().sum()

np.int64(0)

In [ ]:
# check utm_source
raw_tables['website_sessions']['utm_source'].value_counts()

,count
utm_source,
gsearch,316035
bsearch,62823
socialbook,10685


In [ ]:
# check utm_campaign
raw_tables['website_sessions']['utm_campaign'].value_counts()

,count
utm_campaign,
nonbrand,337615
brand,41243
desktop_targeted,5590
pilot,5095


In [ ]:
# check utm_content
raw_tables['website_sessions']['utm_content'].value_counts()

,count
utm_content,
g_ad_1,282706
b_ad_1,54909
g_ad_2,33329
b_ad_2,7914
social_ad_2,5590
social_ad_1,5095


In [ ]:
# check url
raw_tables['website_sessions']['http_referer'].value_counts()

,count
http_referer,
https://www.gsearch.com,351237
https://www.bsearch.com,71032
https://www.socialbook.com,10685


**Observation for Website Sessions Table**

Based on the analysis of the `website_sessions` table, we can make the following observations:

*   **Data Overview**: The table contains 472,871 entries and 9 columns, including `website_session_id`, `created_at`, `user_id`, `is_repeat_session`, `utm_source`, `utm_campaign`, `utm_content`, `device_type`, and `http_referer`.
*   **Data Types**: `website_session_id`, `user_id`, and `is_repeat_session` are integers (`int64`). The other columns (`created_at`, `utm_source`, `utm_campaign`, `utm_content`, `device_type`, `http_referer`) are object types. The `created_at` column should be converted to a datetime object for time-series analysis.
*   **Missing Values**: The `website_sessions` table has significant missing values in several columns:
    *   `utm_source`: 83,328 null values.
    *   `utm_campaign`: 83,328 null values.
    *   `utm_content`: 83,328 null values.
    *   `http_referer`: 39,917 null values.
    These missing values will need to be addressed, potentially through imputation or by understanding why they are missing.
*   **Duplicate Rows**: The `website_sessions` table contains no duplicate rows, ensuring the uniqueness of each session record.

**Data cleaning**

In [ ]:
# Covert data format created_at
raw_tables['website_sessions']['created_at'] = raw_tables['website_sessions']['created_at'].astype('datetime64[ns]')

In [ ]:
traffic_patterns = raw_tables['website_sessions'].groupby(['utm_source', 'utm_campaign']).size().reset_index(name='session_count')
display(traffic_patterns.sort_values(by='session_count', ascending=False))

,utm_source,utm_campaign,session_count
3,gsearch,nonbrand,282706
1,bsearch,nonbrand,54909
2,gsearch,brand,33329
0,bsearch,brand,7914
4,socialbook,desktop_targeted,5590
5,socialbook,pilot,5095


In [ ]:
from urllib.parse import urlparse

sessions_clean = raw_tables['website_sessions'].copy()

sessions_clean["created_at"] = sessions_clean["created_at"].astype("datetime64[ns]")

# Convert text
for col in [
    "utm_source", "utm_campaign", "utm_content",
    "http_referer", "device_type"
]:
    sessions_clean[col] = (
        sessions_clean[col]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

#Separate hostname and referrer
referrer_host = sessions_clean["http_referer"].map(
    lambda url: urlparse(url).hostname if pd.notna(url) else None
)
referrer_source = referrer_host.map({
    "www.gsearch.com": "gsearch",
    "www.bsearch.com": "bsearch",
    "www.socialbook.com": "socialbook"})

# sessions
has_utm = sessions_clean["utm_source"].notna()
use_referrer = ~has_utm & referrer_source.notna()
no_source = (
    ~has_utm
    & sessions_clean["http_referer"].isna()
)

sessions_clean["traffic_source"] = "unknown"
sessions_clean["source_basis"] = "unavailable"

sessions_clean.loc[has_utm, "traffic_source"] = (
    sessions_clean.loc[has_utm, "utm_source"]
)
sessions_clean.loc[has_utm, "source_basis"] = "utm"

sessions_clean.loc[use_referrer, "traffic_source"] = (
    referrer_source.loc[use_referrer]
)
sessions_clean.loc[use_referrer, "source_basis"] = "referrer"

sessions_clean.loc[no_source, "traffic_source"] = "direct_or_unknown"

In [ ]:
source_summary = (
    sessions_clean
    .groupby(["traffic_source", "source_basis"])
    .size()
    .reset_index(name="session_count")
)

display(source_summary)

assert len(sessions_clean) == len(raw_tables['website_sessions'])
assert sessions_clean["website_session_id"].notna().all()
assert sessions_clean["website_session_id"].is_unique
assert source_summary["session_count"].sum() == len(sessions_clean)

,traffic_source,source_basis,session_count
0,bsearch,referrer,8209
1,bsearch,utm,62823
2,direct_or_unknown,unavailable,39917
3,gsearch,referrer,35202
4,gsearch,utm,316035
5,socialbook,utm,10685


In [ ]:
sessions_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 472871 entries, 0 to 472870
Data columns (total 11 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   website_session_id  472871 non-null  Int64         
 1   created_at          472871 non-null  datetime64[ns]
 2   user_id             472871 non-null  Int64         
 3   is_repeat_session   472871 non-null  Int64         
 4   utm_source          389543 non-null  string        
 5   utm_campaign        389543 non-null  string        
 6   utm_content         389543 non-null  string        
 7   device_type         472871 non-null  string        
 8   http_referer        432954 non-null  string        
 9   traffic_source      472871 non-null  object        
 10  source_basis        472871 non-null  object        
dtypes: Int64(3), datetime64[ns](1), object(2), string(5)
memory usage: 41.0+ MB


# **4. Đẩy Data clean lên Big Query**

In [ ]:
clean_tables = {
    "products": products_clean,
    "orders": orders_clean,
    "order_items": order_items_clean,
    "order_item_refunds": order_items_refunds_clean,
    "website_sessions": sessions_clean,
    "website_pageviews": website_pageviews_clean
}

In [ ]:
import pandas as pd
from google.cloud import bigquery

# Thông tin đích
project_id = "jda-k1"
dataset_id = "bigproject_phap_staging"
dataset = client.get_dataset(f"{project_id}.{dataset_id}")

upload_log = []

for table_id, df in clean_tables.items():
    table_ref = f"{project_id}.{dataset_id}.{table_id}"

    # Tránh ghi đè bảng bằng DataFrame rỗng ngoài ý muốn
    if df.empty:
        raise ValueError(f"{table_id} đang rỗng. Kiểm tra lại cleaning.")

    # Cấu hình nạp
    job_config = bigquery.LoadJobConfig(
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
    )

    # Nạp DataFrame đã clean
    job = client.load_table_from_dataframe(
        df,
        table_ref,
        job_config=job_config,
        location=dataset.location
    )
    job.result()

    # Đối chiếu số dòng
    loaded_table = client.get_table(table_ref)

    if loaded_table.num_rows != len(df):
        raise ValueError(f"Số dòng không khớp ở bảng {table_id}")

    upload_log.append({
        "table": table_id,
        "dataframe_rows": len(df),
        "bigquery_rows": loaded_table.num_rows,
        "status": "OK"
    })

    print(f"OK — {table_ref}: {loaded_table.num_rows:,} dòng")

display(pd.DataFrame(upload_log))

OK — jda-k1.bigproject_phap_staging.products: 4 dòng
OK — jda-k1.bigproject_phap_staging.orders: 32,313 dòng
OK — jda-k1.bigproject_phap_staging.order_items: 40,025 dòng
OK — jda-k1.bigproject_phap_staging.order_item_refunds: 1,731 dòng
OK — jda-k1.bigproject_phap_staging.website_sessions: 472,871 dòng
OK — jda-k1.bigproject_phap_staging.website_pageviews: 1,188,124 dòng


,table,dataframe_rows,bigquery_rows,status
0,products,4,4,OK
1,orders,32313,32313,OK
2,order_items,40025,40025,OK
3,order_item_refunds,1731,1731,OK
4,website_sessions,472871,472871,OK
5,website_pageviews,1188124,1188124,OK
